# MICrONS Option 3 — Time-resolved PCA trajectories

Per-area, per-stimulus trial-averaged trajectories with the time axis preserved. Resolves the Monet2/Trippy collapse seen in Option 2 by keeping within-trial dynamics. See `docs/specs/2026-05-04-option3-trajectories-design.md` for design and `docs/plans/2026-05-04-option3-trajectories-implementation.md` for the implementation plan.

## How to use this notebook

Pipeline is **session-swappable**: change `SESSION` in the configuration block below and restart the kernel.

## Part 0 — Setup

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import microns_eda
import option2_pca_utils
import option3_trajectories_utils as traj_utils

In [ ]:
# === Configuration block — every tunable lives here. ===
SESSION = "7_5"

# Preprocessing (reused from Option 2).
N_FRAMES_TRUNCATE = 75
TREADMILL_OUTLIER_THRESHOLD = 1.0

# PCA / metrics.
N_COMPONENTS = 10
N_PCS_FOR_DISTANCE = 3

# Bootstrap / null.
N_BOOT = 1000
N_SHUFFLES = 100

# Subsampling controls.
N_POPULATION_SUBSAMPLES = 20
N_CLIP_SUBSAMPLES = 20

# Reproducibility.
RANDOM_SEED = 42

# Paths.
DATADIR = Path(os.environ.get("MICRONS_DATADIR", "../neuroscience"))
FIGURES_DIR = Path(f"figures/option3/{SESSION}")
RESULTS_DIR = Path(f"results/option3/{SESSION}")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# 7.5 Hz sampling → ~133 ms/frame, ~10 s total.
def frame_to_ms(f):
    return f * 1000.0 / 7.5

np.random.seed(RANDOM_SEED)

print(f"SESSION       = {SESSION}")
print(f"DATADIR       = {DATADIR}")
print(f"FIGURES_DIR   = {FIGURES_DIR}")
print(f"RESULTS_DIR   = {RESULTS_DIR}")
print(f"75 frames     = {frame_to_ms(75):.1f} ms ≈ {frame_to_ms(75)/1000:.1f} s")

### Sanity check

Verifies the data loads correctly, recomputes `clean_trial_indices`, and checks frame-0 alignment via `stim_times`. All session-specific values are read from data (no hard-coded numbers) so this cell adapts to any session.

In [ ]:
# === Sanity check ===
reader = microns_eda.open_dataset(DATADIR)
sessions = microns_eda.list_sessions(DATADIR)
assert SESSION in sessions, f"session {SESSION!r} not in {sessions}"
print(f"Found {len(sessions)} sessions; using {SESSION!r}.")

meta = microns_eda.get_session_meta(DATADIR, SESSION)
print(f"  n_neurons (read from data) = {meta['n_neurons']}")

# Per-trial treadmill mean → cleaning.
n_trials = meta["n_trials"]
per_trial_tread_means = np.empty(n_trials, dtype=np.float64)
for i in range(n_trials):
    trial = microns_eda.load_trial(reader, DATADIR, SESSION, i)
    per_trial_tread_means[i] = np.nanmean(trial["treadmill"])
clean_trial_indices, running_mask = microns_eda.compute_clean_trial_indices(
    per_trial_tread_means, threshold=TREADMILL_OUTLIER_THRESHOLD,
)

stim_map = microns_eda.build_stim_type_map(reader, SESSION)
stim_types_full = np.array([
    stim_map[h.decode() if isinstance(h, bytes) else h]
    for h in meta["condition_hashes"]
])
labels = stim_types_full[clean_trial_indices]
class_counts = pd.Series(labels).value_counts().to_dict()

print(f"  n_trials (total)            = {n_trials}")
print(f"  n_trials (after running QC) = {len(clean_trial_indices)}")
print(f"  per-class breakdown (clean):")
for k in sorted(class_counts):
    print(f"    {k:8s} {class_counts[k]}")

# Frame-0 alignment check via stim_times.
trial0 = microns_eda.load_trial(reader, DATADIR, SESSION, int(clean_trial_indices[0]))
print(f"  sample trial stim_times shape = {trial0['stim_times'].shape}")
print(f"  sample trial stim_times[:3]   = {trial0['stim_times'][:3]}")
print("  (Frame-0 alignment is implicit in the EDA's truncation; no shift needed.)")

assert len(clean_trial_indices) > 0, "no clean trials remain"
assert len(class_counts) >= 2, "need ≥2 stim classes"
print("Sanity check passed.")

## Part 0.5 — PSTH-like sanity check

Per-area, per-stim, per-frame area-mean response (one number per frame, averaged across neurons). Three curves per area on shared axes, no PCA.

This is the rawest temporal view of stimulus differences. We look here first, before any geometric analysis. **What we're spotting:** does the population mean rise within the first frames after stimulus onset? Does it saturate or decay? Do the three stim classes already separate at the per-frame mean level (Option 2 said yes, this confirms it visually)?

In [ ]:
# We need the preprocessed responses + per-area trajectories for PSTH; build them once here
# (Part 1 will use the same outputs).
print("loading + preprocessing responses...")
responses, trial_boundaries, _ = microns_eda.load_session_responses(
    reader, DATADIR, SESSION
)
responses_pp = option2_pca_utils.preprocess_responses(responses, apply_log=False)
trajectories = traj_utils.build_stim_trajectories(
    responses_pp, trial_boundaries, clean_trial_indices,
    meta["brain_areas"], labels, n_frames=N_FRAMES_TRUNCATE,
)
print(f"trajectories built for areas {sorted(trajectories.keys())}")

# Four-panel PSTH (one per area).
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for area, ax in zip(["V1", "AL", "LM", "RL"], axes.flat):
    if area in trajectories:
        traj_utils.plot_psth_per_stim_area(trajectories, area, ax=ax)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "0_5_psth_per_area.png", dpi=150)
plt.show()

## Part 1 — Build per-area, per-stimulus trajectories

We've already built `trajectories` above for the PSTH plot. Here we save them to disk for later reload (so subsequent runs of the notebook can skip the slow preprocessing step) and run sanity asserts.

In [ ]:
# Sanity asserts on the trajectory shapes.
for area, per_stim in trajectories.items():
    for stim, traj in per_stim.items():
        assert traj.shape[0] == N_FRAMES_TRUNCATE, f"{area}/{stim} wrong n_frames"
        assert not np.isnan(traj).any(), f"{area}/{stim} has NaN"
        assert (traj.std(axis=0) > 0).all(), f"{area}/{stim} has zero-var neurons"
print("trajectory sanity asserts passed.")

# Save to disk for fast reload.
traj_path = RESULTS_DIR / "trajectories.npz"
traj_utils.save_trajectories(trajectories, traj_path)
print(f"saved {traj_path}")

## Part 2 — Fit PCA per area on stacked stimulus trajectories

For each area, vertically stack the 3 stimulus matrices into a `(225, n_neurons_in_area)` matrix and fit PCA. The 3 stimuli share a common coordinate system within an area, so trajectories are directly comparable.

**Note: PC1 in this stacked-matrix PCA is typically dominated by stimulus-common temporal dynamics** (rise at onset, settle later) — visualisations should focus on PC2-3 separation as the substantive stimulus-identity finding, not PC1.

In [ ]:
pca_per_area = traj_utils.fit_trajectory_pca(
    trajectories, n_components=N_COMPONENTS, random_state=RANDOM_SEED,
)
print("Per-area PCA fitted:")
for area in sorted(pca_per_area.keys()):
    pca = pca_per_area[area]["pca"]
    top3 = 100 * pca.explained_variance_ratio_[:3].sum()
    top10 = 100 * pca.explained_variance_ratio_[:10].sum()
    print(f"  {area:4s}: top-3 cum var = {top3:5.1f}%   top-10 cum var = {top10:5.1f}%")